In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-11-04 16:11:22--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.2.33, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M  57.4MB/s    in 0.4s    

2026-11-04 16:11:23 (57.4 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [4]:
# add your code here - consider creating a new cell for each section of code
# Count ratings for each user and each book
user_counts = df_ratings['user'].value_counts()
book_counts = df_ratings['isbn'].value_counts()

# Select users with at least 200 ratings
active_users = user_counts[user_counts >= 200].index

# Select books with at least 100 ratings
popular_books = book_counts[book_counts >= 100].index

# Apply both filtering conditions together
df_ratings_filtered = df_ratings[
    (df_ratings['user'].isin(active_users)) &
    (df_ratings['isbn'].isin(popular_books))
]

# Combine ratings with book titles
df_combined = pd.merge(
    df_ratings_filtered,
    df_books,
    on='isbn'
)

# Remove rows without book titles
df_combined = df_combined.dropna(subset=['title'])

# Remove duplicate user-book ratings
df_combined = df_combined.drop_duplicates(['user', 'title'])

# Create the book-user rating matrix
df_pivot = df_combined.pivot(
    index='title',
    columns='user',
    values='rating'
).fillna(0)

# Convert the matrix into a sparse matrix
mat_books = csr_matrix(df_pivot.values)

# Create and train the KNN model
knn = NearestNeighbors(metric='cosine', algorithm='brute')
knn.fit(mat_books)

print("Filtered ratings shape:", df_ratings_filtered.shape)
print("Combined data shape:", df_combined.shape)
print("Rating matrix shape:", df_pivot.shape)
print("KNN model trained successfully!")

Filtered ratings shape: (49781, 3)
Combined data shape: (49136, 5)
Rating matrix shape: (673, 888)
KNN model trained successfully!


In [5]:
# function to return recommended books - this will be tested
def get_recommends(book=""):

    # Find the position of the requested book
    book_index = df_pivot.index.get_loc(book)

    # Find the 6 nearest books (including the requested book)
    distances, indices = knn.kneighbors(
        df_pivot.iloc[book_index, :].values.reshape(1, -1),
        n_neighbors=6
    )

    recommended_books = []

    # Exclude the requested book and collect the other 5 books
    for i in range(1, len(distances[0])):
        recommended_books.append([
            df_pivot.index[indices[0][i]],
            distances[0][i]
        ])

    # Arrange recommendations from largest to smallest distance
    recommended_books.sort(key=lambda x: x[1], reverse=True)

    return [book, recommended_books]

In [6]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", np.float32(0.8016211)], ['The Weight of Water', np.float32(0.77085835)], ['The Surgeon', np.float32(0.7699411)], ['I Know This Much Is True', np.float32(0.7677075)], ['The Lovely Bones: A Novel', np.float32(0.7234864)]]]
You passed the challenge! 🎉🎉🎉🎉🎉
